# abilmansplus/whisper-turbo-ksc2 + chunk / VAD + decloop — 100 аудио

Модель: `abilmansplus/whisper-turbo-ksc2` (Whisper large-v3-turbo, Kazakh Speech Corpus 2-де дообучен).
Екі кесу тәсілі (30 с chunk, VAD), екеуіне де decloop қолданылып, 6 метрика біркелкі нормалаумен есептеледі.
Нәтиже `dataset_wav16k/ksc2/` қалтасына сақталады. Әр файлдан кейін CSV сақталады (resume-safe).
**GPU (T4) қосыңыз:** Runtime → Change runtime type → T4 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install pydub jiwer sacrebleu bert_score transformers accelerate

In [ ]:
import os, time, re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pydub import AudioSegment

DATASET_BASE = "/content/drive/MyDrive/Colab Notebooks/dataset_wav16k"
OUT_DIR = os.path.join(DATASET_BASE, "ksc2")
os.makedirs(OUT_DIR, exist_ok=True)
DURATIONS = ["30s", "1m", "2m", "5m", "10m"]

def collect_pairs(base, durations):
    pairs = []
    for d in durations:
        audio_dir = os.path.join(base, "audio", d)
        ref_dir = os.path.join(base, "references", d)
        if not os.path.isdir(audio_dir):
            print("[WARN] аудио қалтасы жоқ:", audio_dir); continue
        for fname in sorted(os.listdir(audio_dir)):
            if not fname.lower().endswith(".wav"): continue
            stem = os.path.splitext(fname)[0]
            ref_path = os.path.join(ref_dir, stem + ".txt")
            if not os.path.exists(ref_path):
                print("[WARN] REF жоқ:", ref_path); continue
            pairs.append(dict(duration=d, stem=stem,
                              audio_path=os.path.join(audio_dir, fname), ref_path=ref_path))
    return pairs

pairs = collect_pairs(DATASET_BASE, DURATIONS)
print("Жұп саны:", len(pairs))

In [ ]:
import jiwer, sacrebleu, bert_score

def norm_text(t):
    t = str(t).lower()
    t = re.sub(r"[^\w\s]", "", t)
    return re.sub(r"\s+", " ", t).strip()

COMPUTE_TER = True   # ұзын (5m/10m) файлдарда TER баяу (~1 мин/файл). Қаламасаңыз False қойыңыз.

_scorer = bert_score.BERTScorer(lang="other", model_type="bert-base-multilingual-cased")  # бір рет жүктеледі
_ter = sacrebleu.metrics.TER()

def all_metrics(ref, hyp, with_ter=True, with_bert=True):
    r, h = norm_text(ref), norm_text(hyp)
    out = dict(
        wer=jiwer.wer(r, h), cer=jiwer.cer(r, h),
        bleu=sacrebleu.sentence_bleu(h, [r]).score,
        chrf=sacrebleu.sentence_chrf(h, [r]).score,
    )
    out["ter"] = _ter.sentence_score(h, [r]).score if (with_ter and COMPUTE_TER) else float("nan")
    out["bertscore"] = _scorer.score([h], [r])[2].item() if with_bert else float("nan")
    return out

In [ ]:
import torch
from transformers import pipeline

KSC2_MODEL = "abilmansplus/whisper-turbo-ksc2"
device_ok = torch.cuda.is_available()
asr = pipeline(
    "automatic-speech-recognition",
    model=KSC2_MODEL,
    torch_dtype=torch.float16 if device_ok else torch.float32,
    device=0 if device_ok else -1,
)
print("Жүктелді:", KSC2_MODEL, "| device:", "cuda" if device_ok else "cpu  (!!! GPU квотасы жоқ болуы мүмкін — өте баяу)")
GEN_KWARGS = dict(language="kk", task="transcribe")

In [ ]:
def vad_segment(x, sr=16000, win=0.025, min_sil=0.30, max_len=18.0, min_len=1.0, pad=0.25):
    n = int(win * sr)
    if len(x) < n * 2:
        return [(0, len(x))]
    frames = x[: len(x) // n * n].reshape(-1, n)
    e = np.sqrt((frames ** 2).mean(axis=1) + 1e-12)
    floor = np.percentile(e, 10); speech = np.percentile(e, 75)
    thr = floor + 0.12 * max(speech - floor, 1e-6)
    sil = e < thr
    need = max(1, int(min_sil / win))
    cuts, run = [], 0
    for i, s in enumerate(sil):
        if s: run += 1
        else:
            if run >= need: cuts.append((i - run // 2) * n)
            run = 0
    bounds = sorted({0, len(x)} | {c for c in cuts if 0 < c < len(x)})
    segs, a = [], bounds[0]
    for b in bounds[1:]:
        if (b - a) / sr < min_len and b != bounds[-1]:
            continue
        while (b - a) / sr > max_len:
            segs.append((a, a + int(max_len * sr))); a += int(max_len * sr)
        segs.append((a, b)); a = b
    return [(max(0, s - int(pad * sr)), min(len(x), e2 + int(pad * sr))) for s, e2 in segs]

def load_audio(path, sr=16000):
    a = AudioSegment.from_file(path).set_channels(1).set_frame_rate(sr)
    x = np.array(a.get_array_of_samples()).astype(np.float32)
    return x / float(1 << (8 * a.sample_width - 1)), sr


import shutil, tempfile
CHUNK_MS = 30_000

def transcribe_chunk(audio_path):
    audio = AudioSegment.from_file(audio_path)
    tmp = tempfile.mkdtemp(prefix="ksc2_chunks_")
    texts = []
    try:
        for start in range(0, max(len(audio), 1), CHUNK_MS):
            p = os.path.join(tmp, f"c_{start:08d}.wav")
            audio[start:start + CHUNK_MS].export(p, format="wav")
            texts.append(str(asr(p, generate_kwargs=GEN_KWARGS)["text"]).strip())
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    return " ".join(t for t in texts if t)

def transcribe_vad(audio_path):
    x, sr = load_audio(audio_path)
    texts = []
    for s, e in vad_segment(x, sr):
        c = x[s:e]
        if len(c) < sr * 0.2: continue
        texts.append(str(asr({"array": c, "sampling_rate": sr}, generate_kwargs=GEN_KWARGS)["text"]).strip())
    return " ".join(t for t in texts if t)

In [ ]:
# Тест: бір файл
p = pairs[0]
t0 = time.time()
ref0 = open(p["ref_path"], encoding="utf-8").read().strip()
r_chunk = transcribe_chunk(p["audio_path"]); r_vad = transcribe_vad(p["audio_path"])
print("уақыт:", round(time.time()-t0, 1), "сек")
print("REF  :", ref0[:200]); print("chunk:", r_chunk[:200]); print("VAD  :", r_vad[:200])
print(all_metrics(ref0, r_chunk, False, False)); print(all_metrics(ref0, r_vad, False, False))

In [ ]:
def run_batch(fn, out_name, desc):
    out_csv = os.path.join(OUT_DIR, out_name)
    done, results = set(), []
    if os.path.exists(out_csv):
        prev = pd.read_csv(out_csv)
        results = prev.to_dict("records"); done = set(zip(prev["duration"], prev["stem"]))
        print(out_name, ": бұрынғы", len(results), "жол өткізіледі")
    t0 = time.time()
    for item in tqdm(pairs, desc=desc):
        key = (item["duration"], item["stem"])
        if key in done: continue
        try:
            ref = open(item["ref_path"], encoding="utf-8").read().strip()
            raw = fn(item["audio_path"])
            row = dict(duration=item["duration"], stem=item["stem"], ref=ref, raw=raw, fixed=raw)
            row.update({f"{k}_raw": v for k, v in all_metrics(ref, raw, with_ter=False, with_bert=False).items()})
            results.append(row); done.add(key)
        except Exception as e:
            print("[ERROR]", item["duration"], item["stem"], e); continue
        pd.DataFrame(results).to_csv(out_csv, index=False)   # ӘР файлдан кейін сақтау
    pd.DataFrame(results).to_csv(out_csv, index=False)
    print(out_name, ": аяқталды", len(results), "/", len(pairs), "|", round((time.time()-t0)/60, 1), "мин")
    return out_csv

CSV_CHUNK = run_batch(transcribe_chunk, "mapp_v2_batch_results_ksc2_chunk.csv", "ksc2 + chunk")

In [ ]:
CSV_VAD = run_batch(transcribe_vad, "mapp_v2_batch_results_ksc2_vad.csv", "ksc2 + VAD")

In [ ]:
def collapse_loops(text, max_run=2):
    out, prev, run = [], None, 0
    for w in str(text).split():
        if w == prev:
            run += 1
            if run <= max_run: out.append(w)
        else:
            prev, run = w, 1; out.append(w)
    return " ".join(out)

def full_eval(df, label):
    """raw және decloop үшін 6 метриканың орташасы (біркелкі нормалау)."""
    out = {}
    for tag, col in [(label + " raw", "raw"), (label + "+decloop", "raw_decloop")]:
        ms = [all_metrics(r, h) for r, h in tqdm(list(zip(df["ref"], df[col])), desc=tag)]
        out[tag] = pd.DataFrame(ms).mean()
    return out

dfc = pd.read_csv(CSV_CHUNK); dfc["raw_decloop"] = dfc["raw"].apply(collapse_loops)
dfv = pd.read_csv(CSV_VAD);   dfv["raw_decloop"] = dfv["raw"].apply(collapse_loops)

cols = {}
cols.update(full_eval(dfc, "ksc2 chunk"))
cols.update(full_eval(dfv, "ksc2 VAD"))
cmp_df = pd.DataFrame(cols).round(4)
display(cmp_df)

# Салыстыру үшін: бұрынғы Shyngys (kk-turbo) нәтижелері, егер CSV бар болса (тек жылдам 4 метрика)
def fast(df, col):
    return pd.DataFrame([all_metrics(r, h, False, False) for r, h in zip(df["ref"], df[col])]).mean()
ref_cols = {}
for name, fn in [("kk-turbo chunk", "mapp_v2_batch_results_kkturbo_chunk.csv"), ("kk-turbo VAD", "mapp_v2_batch_results_kkturbo_vad.csv")]:
    pth = os.path.join(DATASET_BASE, fn)
    if os.path.exists(pth):
        d = pd.read_csv(pth); d["raw_decloop"] = d["raw"].apply(collapse_loops)
        ref_cols[name + " raw"] = fast(d, "raw"); ref_cols[name + "+decloop"] = fast(d, "raw_decloop")
if ref_cols:
    print("\nБұрынғы kk-turbo (Shyngys) — жылдам 4 метрика, біркелкі нормалау:")
    display(pd.DataFrame(ref_cols).round(4))

In [ ]:
OUT_XLSX = os.path.join(OUT_DIR, "mapp_asr_ksc2_comparison.xlsx")
_bad = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
clean = lambda v: _bad.sub("", v) if isinstance(v, str) else v
with pd.ExcelWriter(OUT_XLSX) as w:
    cmp_df.to_excel(w, sheet_name="summary_6metrics")
    if ref_cols: pd.DataFrame(ref_cols).round(4).to_excel(w, sheet_name="kkturbo_reference")
    dfc.map(clean).to_excel(w, sheet_name="ksc2_chunk_per_file", index=False)
    dfv.map(clean).to_excel(w, sheet_name="ksc2_vad_per_file", index=False)
print("Excel сақталды:", OUT_XLSX)